# APIndex Weight Sensitivity Analysis

## Purpose

This notebook tests the **model-level robustness** of the Action Priority Index (APIndex) after the distribution and threshold assessment in Part 1.

It answers three questions:

1. Can the Tableau APIndex be reproduced exactly from the normalized KPI inputs and stakeholder weights?
2. How strongly do APIndex values and customer rankings react to plausible changes in KPI weights?
3. How stable are the candidate **Action Priority thresholds** (`Medium ≥ 0.30`, `High ≥ 0.60`) under those changes?

This is intentionally separate from the earlier statistical validation of the source data. The statistical notebook evaluates **data-level evidence**; this notebook evaluates **decision-model robustness**.

### Baseline stakeholder profiles

| Perspective | Freight | Capacity | Delivery |
|---|---:|---:|---:|
| Strategic | 20% | 50% | 30% |
| Tactical | 40% | 20% | 40% |
| Operational | 20% | 20% | 60% |

The APIndex follows the Tableau implementation:

\[
APIndex = \frac{F \cdot w_F + C \cdot w_C + D \cdot w_D}
{w_F + w_C + w_D}
\]

where `F`, `C`, and `D` are normalized KPI inputs between 0 and 1.


### 1. Load model inputs and Tableau baseline exports

`api_model_inputs.csv` is exported from Tableau in long format (`Measure Names` / `Measure Values`).  
The code pivots it to one row per customer and reads the three existing Tableau APIndex exports for validation.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

DATA_DIR = Path("processed_data")

INPUT_FILE = DATA_DIR / "api_model_inputs.csv"

APINDEX_FILES = {
    "Strategic": DATA_DIR / "api_strategic.csv",
    "Tactical": DATA_DIR / "api_tactical.csv",
    "Operational": DATA_DIR / "api_operational.csv",
}

WEIGHTS = {
    "Strategic": {"freight": 20, "capacity": 50, "delivery": 30},
    "Tactical": {"freight": 40, "capacity": 20, "delivery": 40},
    "Operational": {"freight": 20, "capacity": 20, "delivery": 60},
}

KPI_COLUMNS = {
    "freight": "Freight Cost Exposure Norm",
    "capacity": "Capacity Risk Norm",
    "delivery": "Delivery Risk Norm",
}

MEDIUM_THRESHOLD = 0.30
HIGH_THRESHOLD = 0.60

def read_tableau_csv(path):
    return pd.read_csv(
        path,
        encoding="utf-16",
        sep="\t",
        decimal=","
    )

inputs_long = read_tableau_csv(INPUT_FILE)

inputs = (
    inputs_long
    .pivot(index="Customer Label", columns="Measure Names", values="Measure Values")
    .reset_index()
)
inputs.columns.name = None

inputs.head()


,Customer Label,Capacity Risk Norm,Delivery Risk Norm,Freight Cost Exposure Norm
0,C01,0.998936,0.0,0.003535
1,C02,1.000000,0.0,0.000132
2,C03,1.000000,0.0,0.000000
3,C04,1.000000,0.0,0.177707
4,C05,0.990159,NaN,0.003249


### Completeness check

The sensitivity analysis requires all three normalized KPI inputs. Customers with an incomplete KPI basis are retained in the source but excluded from weight perturbation calculations.


In [2]:
required = list(KPI_COLUMNS.values())

completeness = pd.DataFrame({
    "customers_total": [len(inputs)],
    "customers_complete": [inputs[required].notna().all(axis=1).sum()],
    "customers_incomplete": [inputs[required].isna().any(axis=1).sum()],
})

display(completeness)

incomplete_customers = inputs.loc[
    inputs[required].isna().any(axis=1),
    ["Customer Label"] + required
]

display(incomplete_customers)

model = inputs.dropna(subset=required).copy()


,customers_total,customers_complete,customers_incomplete
0,46,40,6


,Customer Label,Freight Cost Exposure Norm,Capacity Risk Norm,Delivery Risk Norm
4,C05,0.003249,0.990159,NaN
17,C18,NaN,0.000000,0.0
25,C26,NaN,1.000000,0.0
33,C34,NaN,1.000000,0.0
35,C36,NaN,1.000000,0.0
37,C38,0.025599,1.000000,NaN


**Observed result:** the model-input export contains 46 customers, of which **40 have a complete three-KPI basis**. This matches the valid-customer basis used in Part 1. The sensitivity results below therefore refer to these 40 customers.


### 2. Baseline reproduction check

Before changing any weights, the Python implementation should reproduce the Tableau APIndex. This verifies that the sensitivity analysis is testing the **same calculation**, rather than a separate approximation.


In [3]:
def calculate_apindex(df, weights):
    total_weight = sum(weights.values())

    return (
        df[KPI_COLUMNS["freight"]] * weights["freight"]
        + df[KPI_COLUMNS["capacity"]] * weights["capacity"]
        + df[KPI_COLUMNS["delivery"]] * weights["delivery"]
    ) / total_weight


reproduction_rows = []

for perspective, weights in WEIGHTS.items():
    calc = model[["Customer Label"]].copy()
    calc["python_apindex"] = calculate_apindex(model, weights)

    tableau = read_tableau_csv(APINDEX_FILES[perspective]).rename(
        columns={"APIndex Total": "tableau_apindex"}
    )
    tableau["tableau_apindex"] = pd.to_numeric(
        tableau["tableau_apindex"], errors="coerce"
    )

    check = calc.merge(
        tableau[["Customer Label", "tableau_apindex"]],
        on="Customer Label",
        how="left"
    )

    check["abs_difference"] = (
        check["python_apindex"] - check["tableau_apindex"]
    ).abs()

    reproduction_rows.append({
        "perspective": perspective,
        "n": len(check),
        "max_abs_difference": check["abs_difference"].max(),
        "mean_abs_difference": check["abs_difference"].mean(),
    })

reproduction_check = pd.DataFrame(reproduction_rows)
reproduction_check


,perspective,n,max_abs_difference,mean_abs_difference
0,Strategic,40,6.000000e-10,1.950000e-10
1,Tactical,40,6.000000e-10,2.500000e-10
2,Operational,40,6.000000e-10,2.550000e-10


### Interpretation

The reproduction check should show differences only around floating-point precision (approximately `10⁻¹⁰` with the current exports). This confirms that the Python implementation reproduces the Tableau APIndex effectively exactly.

**Implication:** subsequent changes can be interpreted as genuine effects of altered KPI weights, not differences between Tableau and Python implementations.


### 3. Sensitivity scenario design

A **±10 percentage-point** change is applied to one KPI weight at a time.

To keep the total weight at 100%, the remaining two weights are adjusted **proportionally**. This preserves their relative relationship instead of arbitrarily assigning the balancing change to only one KPI.

Example — Strategic Capacity:

- Baseline: `Freight 20 / Capacity 50 / Delivery 30`
- Capacity +10 pp: `16 / 60 / 24`
- Capacity −10 pp: `24 / 40 / 36`

Each perspective therefore has six perturbation scenarios plus its baseline.


In [4]:
def perturb_weights(base_weights, target, delta_pp):
    new_weights = base_weights.copy()

    new_target = new_weights[target] + delta_pp
    if not 0 <= new_target <= 100:
        raise ValueError("Perturbed target weight must remain between 0 and 100.")

    other_keys = [k for k in new_weights if k != target]
    old_other_total = sum(new_weights[k] for k in other_keys)
    new_other_total = 100 - new_target

    new_weights[target] = new_target

    for key in other_keys:
        new_weights[key] = (
            base_weights[key] / old_other_total
        ) * new_other_total

    return new_weights


scenario_rows = []

for perspective, base_weights in WEIGHTS.items():
    scenario_rows.append({
        "perspective": perspective,
        "scenario": "Baseline",
        **base_weights
    })

    for target in ["freight", "capacity", "delivery"]:
        for delta in [-10, 10]:
            scenario_weights = perturb_weights(base_weights, target, delta)
            scenario_rows.append({
                "perspective": perspective,
                "scenario": f"{target.title()} {delta:+d}pp",
                **scenario_weights
            })

scenario_weights = pd.DataFrame(scenario_rows)
scenario_weights.round(2)


,perspective,scenario,freight,capacity,delivery
0,Strategic,Baseline,20.00,50.00,30.00
1,Strategic,Freight -10pp,10.00,56.25,33.75
2,Strategic,Freight +10pp,30.00,43.75,26.25
3,Strategic,Capacity -10pp,24.00,40.00,36.00
4,Strategic,Capacity +10pp,16.00,60.00,24.00
5,Strategic,Delivery -10pp,22.86,57.14,20.00
6,Strategic,Delivery +10pp,17.14,42.86,40.00
7,Tactical,Baseline,40.00,20.00,40.00
8,Tactical,Freight -10pp,30.00,23.33,46.67
9,Tactical,Freight +10pp,50.00,16.67,33.33


### 4. APIndex movement and ranking stability

Two complementary robustness measures are used:

- **APIndex movement:** mean and maximum absolute change from the baseline.
- **Spearman rank correlation:** whether the customer ordering remains similar even when APIndex values move.

A high rank correlation can coexist with threshold switches. This distinction is important: the ranking may remain stable while customers near `0.30` or `0.60` cross an Action Priority boundary.


In [5]:
def classify_priority(series):
    return pd.cut(
        series,
        bins=[-np.inf, MEDIUM_THRESHOLD, HIGH_THRESHOLD, np.inf],
        labels=["Low", "Medium", "High"],
        right=False
    )


sensitivity_rows = []
customer_scenarios = []

for perspective, base_weights in WEIGHTS.items():
    baseline = calculate_apindex(model, base_weights)
    baseline_class = classify_priority(baseline)

    for target in ["freight", "capacity", "delivery"]:
        for delta in [-10, 10]:
            scenario_weights_dict = perturb_weights(
                base_weights, target, delta
            )
            scenario = calculate_apindex(model, scenario_weights_dict)
            scenario_class = classify_priority(scenario)

            rho, _ = spearmanr(baseline, scenario)

            sensitivity_rows.append({
                "perspective": perspective,
                "scenario": f"{target.title()} {delta:+d}pp",
                "mean_abs_change": (scenario - baseline).abs().mean(),
                "max_abs_change": (scenario - baseline).abs().max(),
                "spearman_rho": rho,
                "priority_switches": (scenario_class != baseline_class).sum(),
            })

            temp = pd.DataFrame({
                "customer": model["Customer Label"].values,
                "perspective": perspective,
                "scenario": f"{target.title()} {delta:+d}pp",
                "baseline_apindex": baseline.values,
                "scenario_apindex": scenario.values,
                "baseline_priority": baseline_class.astype(str).values,
                "scenario_priority": scenario_class.astype(str).values,
            })
            temp["abs_change"] = (
                temp["scenario_apindex"] - temp["baseline_apindex"]
            ).abs()

            customer_scenarios.append(temp)

sensitivity_summary = pd.DataFrame(sensitivity_rows)
customer_sensitivity = pd.concat(customer_scenarios, ignore_index=True)

sensitivity_summary.round(4)


,perspective,scenario,mean_abs_change,max_abs_change,spearman_rho,priority_switches
0,Strategic,Freight -10pp,0.0544,0.0998,0.9976,0
1,Strategic,Freight +10pp,0.0544,0.0998,0.9730,0
2,Strategic,Capacity -10pp,0.0876,0.1000,0.9769,0
3,Strategic,Capacity +10pp,0.0876,0.1000,0.9983,21
4,Strategic,Delivery -10pp,0.0658,0.0984,0.9998,4
5,Strategic,Delivery +10pp,0.0658,0.0984,1.0000,1
6,Tactical,Freight -10pp,0.0299,0.0998,0.9983,0
7,Tactical,Freight +10pp,0.0299,0.0998,0.9961,2
8,Tactical,Capacity -10pp,0.0873,0.1000,0.9885,1
9,Tactical,Capacity +10pp,0.0873,0.1000,0.9968,27


### Observed pattern

With the current model inputs, rank correlations remain very high across the tested scenarios, indicating that the **overall customer ordering is comparatively robust**.

However, Action Priority classifications are less stable in some scenarios. The strongest effect occurs when **Capacity is increased by 10 percentage points**:

- Strategic: **21** priority switches
- Tactical: **27** priority switches
- Operational: **24** priority switches

This is consistent with the earlier structural finding that Capacity Risk has limited customer-level differentiation: many customers share a very similar or identical capacity score. Increasing its weight can therefore shift a large block of customers together across a fixed threshold.

This does **not** mean that Capacity should be removed. It means that the model is particularly sensitive to assigning additional weight to a structurally concentrated KPI.


In [ ]:
# Visual comparison of classification switches
plot_data = sensitivity_summary.sort_values(
    ["perspective", "priority_switches"],
    ascending=[True, False]
)

for perspective in ["Strategic", "Tactical", "Operational"]:
    subset = plot_data[plot_data["perspective"] == perspective]

    plt.figure(figsize=(9, 4.5))
    plt.bar(subset["scenario"], subset["priority_switches"])
    plt.title(f"Action Priority switches — {perspective}")
    plt.ylabel("Customers changing priority class")
    plt.xlabel("Weight perturbation scenario")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


### 5. Customer-level stability

The next table identifies customers that change Action Priority at least once under the tested perturbations. This separates customers that are robustly positioned inside a class from those that sit close to a decision boundary.


In [6]:
switches = customer_sensitivity[
    customer_sensitivity["baseline_priority"]
    != customer_sensitivity["scenario_priority"]
].copy()

switch_count = (
    switches
    .groupby(["customer", "perspective"])
    .size()
    .rename("n_switches")
    .reset_index()
    .sort_values(["n_switches", "customer"], ascending=[False, True])
)

switch_count.head(20)


,customer,perspective,n_switches
48,C29,Tactical,3
74,C46,Tactical,3
9,C04,Strategic,2
10,C04,Tactical,2
55,C35,Strategic,2
56,C35,Tactical,2
58,C39,Operational,2
59,C39,Strategic,2
60,C39,Tactical,2
68,C42,Strategic,2


### 6. Boundary check: C29 and C46

Part 1 identified C29 and C46 as especially useful validation cases:

- **C29** is High in all three baseline perspectives.
- **C46** is High in Strategic, but Medium in Tactical and Operational; its Tactical baseline (`≈ 0.595`) lies immediately below the candidate High threshold of `0.60`.

The sensitivity test now checks whether these classifications are structurally stable or dependent on relatively small weighting choices.


In [7]:
focus = customer_sensitivity[
    customer_sensitivity["customer"].isin(["C29", "C46"])
].copy()

focus["movement"] = (
    focus["scenario_apindex"] - focus["baseline_apindex"]
)

focus[
    [
        "customer",
        "perspective",
        "scenario",
        "baseline_apindex",
        "scenario_apindex",
        "movement",
        "baseline_priority",
        "scenario_priority",
    ]
].sort_values(
    ["customer", "perspective", "scenario"]
).round(4)


,customer,perspective,scenario,baseline_apindex,scenario_apindex,movement,baseline_priority,scenario_priority
625,C29,Operational,Capacity +10pp,0.8003,0.8253,0.0250,High,High
585,C29,Operational,Capacity -10pp,0.8003,0.7754,-0.0250,High,High
705,C29,Operational,Delivery +10pp,0.8003,0.8503,0.0499,High,High
665,C29,Operational,Delivery -10pp,0.8003,0.7504,-0.0499,High,High
545,C29,Operational,Freight +10pp,0.8003,0.7005,-0.0998,High,High
505,C29,Operational,Freight -10pp,0.8003,0.9002,0.0998,High,High
145,C29,Strategic,Capacity +10pp,0.8003,0.8403,0.0399,High,High
105,C29,Strategic,Capacity -10pp,0.8003,0.7604,-0.0399,High,High
225,C29,Strategic,Delivery +10pp,0.8003,0.8289,0.0285,High,High
185,C29,Strategic,Delivery -10pp,0.8003,0.7718,-0.0285,High,High


### Interpretation of C29 and C46

**C29**

C29 remains High under all tested Strategic and Operational scenarios, but its Tactical classification is less robust. Under some Tactical perturbations it falls below `0.60` into Medium. The customer is therefore a strong priority signal overall, but the statement **“C29 is unconditionally High regardless of weighting” would be too strong**.

**C46**

C46 behaves exactly like a useful boundary case should:

- Strategic is generally High, but one tested scenario can move it just below `0.60`.
- Tactical starts at approximately `0.595`, so modest weight changes can move it between Medium and High.
- Operational remains Medium in most scenarios, but a stronger Delivery weighting can move it just below `0.30`.

This confirms the Part 1 concern: **C46's classification is threshold-sensitive**, especially around the Tactical `0.60` boundary. Its APIndex should therefore be interpreted together with the underlying KPI profile rather than as a categorical truth.


### 7. Threshold robustness

The candidate thresholds are intentionally common across all stakeholder perspectives:

- **Low:** `< 0.30`
- **Medium:** `0.30–<0.60`
- **High:** `≥ 0.60`

The sensitivity analysis does not search for thresholds that minimize switches. Doing so would risk fitting the decision boundaries to this single snapshot.

Instead, the test asks whether the candidate thresholds remain interpretable when plausible stakeholder weights change.


In [8]:
threshold_robustness = (
    sensitivity_summary
    .groupby("perspective")
    .agg(
        scenarios=("scenario", "count"),
        avg_priority_switches=("priority_switches", "mean"),
        max_priority_switches=("priority_switches", "max"),
        min_rank_correlation=("spearman_rho", "min"),
        avg_apindex_change=("mean_abs_change", "mean"),
        max_observed_apindex_change=("max_abs_change", "max"),
    )
    .reset_index()
)

threshold_robustness.round(4)


,perspective,scenarios,avg_priority_switches,max_priority_switches,min_rank_correlation,avg_apindex_change,max_observed_apindex_change
0,Operational,6,4.3333,24,0.9728,0.0532,0.1
1,Strategic,6,4.3333,21,0.9730,0.0693,0.1
2,Tactical,6,6.0000,27,0.9885,0.0507,0.1


## Conclusion

The sensitivity analysis supports a **qualified robustness** interpretation of the APIndex.

### What appears robust

- The Python model reproduces the Tableau APIndex to floating-point precision.
- Customer rankings remain highly correlated under the tested ±10 percentage-point weight changes.
- The stakeholder weighting profiles therefore do not arbitrarily reorder the entire customer base.

### What is sensitive

- Fixed Action Priority classes can change even when ranking remains stable.
- Capacity-weight increases produce the largest number of class switches.
- This sensitivity is consistent with the previously identified Capacity Risk concentration and should be treated as a structural model limitation.
- C46 confirms that customers close to `0.30` or `0.60` can change category under plausible weighting changes.
- C29 is comparatively robust overall, but its Tactical High classification is not invariant.

### Decision-model implication

The candidate `0.30 / 0.60` thresholds remain useful as **decision-support boundaries**, but they should not be interpreted as statistically estimated cut-offs or absolute risk categories.

> **The APIndex ranking is broadly robust to moderate weight changes, while categorical Action Priority is more sensitive near the selected decision thresholds. Sensitivity is strongest when additional weight is assigned to the structurally concentrated Capacity KPI. The index should therefore be used as a prioritization aid together with its KPI-level context, rather than as a standalone deterministic classification.**

This preserves the intended role of the APIndex: a transparent stakeholder-oriented decision-support model rather than a predictive score.
